# Fundamentals of Machine Learning – Programming Assignment
## Heart Disease Prediction using Logistic Regression and Support Vector Machine (SVM)

**Dataset:** Cleveland Heart Disease Dataset
**Source:** https://archive.ics.uci.edu/dataset/45/heart+disease
**Algorithms:** Logistic Regression | Support Vector Machine (SVM)
**Language:** Python | Environment: Jupyter Notebook

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Scikit-learn: models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

---
## 2. Load the Dataset

The **Cleveland Heart Disease Dataset** contains 303 patient records with 13 clinical features.
The target variable indicates the presence (1) or absence (0) of heart disease.

**Dataset Link:** https://archive.ics.uci.edu/dataset/45/heart+disease

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

column_names = [
    'age',        # Age in years
    'sex',        # 1 = male, 0 = female
    'cp',         # Chest pain type (0-3)
    'trestbps',   # Resting blood pressure (mm Hg)
    'chol',       # cholesterol (mg/dl)
    'fbs',        # Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
    'restecg',    # Resting ECG results (0-2)
    'thalach',    # Maximum heart rate achieved
    'exang',      # Exercise-induced angina (1 = yes, 0 = no)
    'oldpeak',    # ST depression induced by exercise
    'slope',      # Slope of peak exercise ST segment (0-2)
    'ca',         # Number of major vessels (0-3) colored by fluoroscopy
    'thal',       # Thalassemia type (3 = normal, 6 = fixed defect, 7 = reversible defect)
    'target'      # Diagnosis (0 = no disease, 1-4 = disease present)
]

# Replace '?' with NaN on load
df = pd.read_csv(url, names=column_names, na_values='?')

# Convert target to binary:  1 = disease0 = no disease,
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("=== Dataset Info ===")
df.info()
print("\n=== Statistical Summary ===")
df.describe()

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], target_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Class distribution:\n{target_counts}")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(10, 5))
df[df['target']==0]['age'].hist(bins=20, alpha=0.7, label='No Disease', color='#2ecc71')
df[df['target']==1]['age'].hist(bins=20, alpha=0.7, label='Disease', color='#e74c3c')
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by Heart Disease Status')
plt.legend()
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Step 1: Handle missing values using median imputation
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print(f"Missing values after imputation: {df_imputed.isnull().sum().sum()}")

# Step 2: Separate features and target
X = df_imputed.drop('target', axis=1)
y = df_imputed['target']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Feature names:  {list(X.columns)}")

In [ ]:
# Step 3: Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")
print(f"Train class distribution: {dict(y_train.value_counts())}")
print(f"Test class distribution:  {dict(y_test.value_counts())}")

In [ ]:
# Step 4: Feature Scaling (StandardScaler - important for SVM and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Only transform test (no data leakage)

print("✅ Feature scaling complete.")
print(f"Mean of scaled training data (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Data PreprocessingStd  of scaled training data (should be ~1): {X_train_scaled.std():.4f}")

---
## 5. Algorithm 1 – Logistic Regression

### Background
Logistic Regression is a statistical classification algorithm that models the probability of a binary outcome using the logistic (sigmoid) function. Despite its name, it is a **classification** algorithm, not regression. It is widely used in medical diagnosis because its coefficients are interpretable.

**Why chosen:** Logistic Regression is well-suited for binary classification problems like heart disease prediction. It handles linearly separable data efficiently, produces probability outputs, and is highly interpretable.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_y_pred     = lr_model.predict(X_test_scaled)
lr_y_prob     = lr_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
lr_accuracy   = accuracy_score(y_test, lr_y_pred)
lr_precision  = precision_score(y_test, lr_y_pred)
lr_recall     = recall_score(y_test, lr_y_pred)
lr_f1         = f1_score(y_test, lr_y_pred)
lr_roc_auc    = roc_auc_score(y_test, lr_y_prob)

print("=== Logistic Regression Results ===")
print(f"Accuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"ROC-AUC:   {lr_roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
# Cross-validation for Logistic Regression
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"Logistic Regression 5-Fold CV Accuracy: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")
print(f"Individual folds: {np.round(lr_cv_scores, 4)}")

In [ ]:
# Confusion matrix for Logistic Regression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm_lr = confusion_matrix(y_test, lr_y_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title('Logistic Regression – Confusion Matrix', fontsize=12)
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance (coefficients)
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_[0]})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=True).index)
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
axes[1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
axes[1].set_title('Logistic Regression – Feature Coefficients', fontsize=12)
axes[1].set_xlabel('Coefficient Value')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('lr_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Algorithm 2 – Support Vector Machine (SVM)

### Background
Support Vector Machine (SVM) is a powerful supervised learning algorithm that finds the optimal hyperplane to separate classes by maximizing the margin between support vectors. The **RBF (Radial Basis Function) kernel** allows SVM to capture non-linear boundaries in the data.

**Why chosen:** SVM with an RBF kernel is effective for medical datasets where relationships between features and the outcome may be non-linear. SVM is also robust to high-dimensional spaces and avoids overfitting with the right regularization parameter (C).

In [ ]:
# Hyperparameter tuning for SVM using GridSearchCV
print("Tuning SVM hyperparameters (this may take a moment)...")

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print(f"Best SVM Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy:    {grid_search.best_score_:.4f}")

In [ ]:
# Train best SVM model
svm_model = grid_search.best_estimator_

# Predictions
svm_y_pred  = svm_model.predict(X_test_scaled)
svm_y_prob  = svm_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
svm_accuracy  = accuracy_score(y_test, svm_y_pred)
svm_precision = precision_score(y_test, svm_y_pred)
svm_recall    = recall_score(y_test, svm_y_pred)
svm_f1        = f1_score(y_test, svm_y_pred)
svm_roc_auc   = roc_auc_score(y_test, svm_y_prob)

print("=== SVM Results ===")
print(f"Accuracy:  {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall:    {svm_recall:.4f}")
print(f"F1 Score:  {svm_f1:.4f}")
print(f"ROC-AUC:   {svm_roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, svm_y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
# Confusion matrix for SVM
plt.figure(figsize=(6, 5))
cm_svm = confusion_matrix(y_test, svm_y_pred)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('SVM – Confusion Matrix', fontsize=12)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Comparison of Algorithms

In [ ]:
# Summary comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Logistic Regression': [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_roc_auc],
    'SVM': [svm_accuracy, svm_precision, svm_recall, svm_f1, svm_roc_auc]
})
comparison_df = comparison_df.set_index('Metric')
comparison_df = comparison_df.round(4)
print("=== Algorithm Comparison ===")
print(comparison_df.to_string())

In [ ]:
# Grouped bar chart: metric comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Bar chart ---
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
lr_vals  = [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_roc_auc]
svm_vals = [svm_accuracy, svm_precision, svm_recall, svm_f1, svm_roc_auc]

x = np.arange(len(metrics))
width = 0.35

bars1 = axes[0].bar(x - width/2, lr_vals,  width, label='Logistic Regression', color='steelblue',  alpha=0.85)
bars2 = axes[0].bar(x + width/2, svm_vals, width, label='SVM',                 color='darkorange', alpha=0.85)

axes[0].set_ylim(0, 1.1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_ylabel('Score')
axes[0].set_title('Algorithm Performance Comparison', fontsize=13)
axes[0].legend()

# Value labels on bars
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# --- ROC Curve ---
lr_fpr,  lr_tpr,  _ = roc_curve(y_test, lr_y_prob)
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_y_prob)

axes[1].plot(lr_fpr,  lr_tpr,  label=f'Logistic Regression (AUC={lr_roc_auc:.3f})',
             color='steelblue', linewidth=2)
axes[1].plot(svm_fpr, svm_tpr, label=f'SVM (AUC={svm_roc_auc:.3f})',
             color='darkorange', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves – LR vs SVM', fontsize=13)
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final summary printout
print("=" * 55)
print("        FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"{'Metric':<20} {'Logistic Reg':>15} {'SVM':>15}")
print("-" * 55)
for m, lr_v, svm_v in zip(metrics, lr_vals, svm_vals):
    print(f"{m:<20} {lr_v:>15.4f} {svm_v:>15.4f}")
print("=" * 55)
winner = 'Logistic Regression' if lr_f1 >= svm_f1 else 'SVM'
print(f"Best F1 Score: {winner}")